# ⚙️ Phase 1 — Preprocessing : NASA C-MAPSS Dataset (FD001)

**Objectif :** Préparer les données pour l'entraînement du modèle LSTM.

---
### 📋 Table des matières
1. Setup & Chargement
2. Calcul du RUL + cap à 125
3. Suppression des capteurs non-informatifs
4. Normalisation MinMaxScaler (fit sur train uniquement)
5. Sliding Window (W=30)
6. Sauvegarde des arrays + scaler

## 1. Setup & Chargement

In [13]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/industrial-ai-platform'
DATA_DIR = f'{BASE_DIR}/CMAPSSData'
OUTPUT_DIR  = f'{BASE_DIR}/phase_preprocessing'
CONFIG_PATH = f'{BASE_DIR}/feature_config.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = f'{DATA_DIR}/train_FD001.txt'
TEST_PATH  = f'{DATA_DIR}/test_FD001.txt'
RUL_PATH   = f'{DATA_DIR}/RUL_FD001.txt'

COLUMNS = [
    'unit_id', 'time_cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    *[f'sensor_{i:02d}' for i in range(1, 22)]
]

df_train = pd.read_csv(TRAIN_PATH, sep=r'\s+', header=None, names=COLUMNS)
df_test  = pd.read_csv(TEST_PATH,  sep=r'\s+', header=None, names=COLUMNS)
df_rul   = pd.read_csv(RUL_PATH,   header=None, names=['RUL_true'])

print(f'Train : {df_train.shape}  |  Test : {df_test.shape}  |  RUL ground truth : {len(df_rul)}')

Train : (20631, 26)  |  Test : (13096, 26)  |  RUL ground truth : 100


In [6]:
# ── Charger la config issue de l'EDA ───────────────────────────────────────
with open(CONFIG_PATH) as f:
    config = json.load(f)

SENSORS_TO_KEEP = config['sensors_to_keep']
SENSORS_TO_DROP = config['sensors_to_drop']
RUL_CAP         = config['rul_cap']           # 125

print(f'RUL cap        : {RUL_CAP}')
print(f'Capteurs retenus ({len(SENSORS_TO_KEEP)}) : {SENSORS_TO_KEEP}')
print(f'Capteurs supprimés ({len(SENSORS_TO_DROP)}) : {SENSORS_TO_DROP}')

RUL cap        : 125
Capteurs retenus (14) : ['sensor_02', 'sensor_03', 'sensor_04', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
Capteurs supprimés (7) : ['sensor_01', 'sensor_05', 'sensor_06', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


## 2. Calcul du RUL + cap à 125

In [7]:
# ── Train : RUL = max_cycle - time_cycle, plafonné à 125 ───────────────────
max_cycles   = df_train.groupby('unit_id')['time_cycle'].max().rename('max_cycle')
df_train     = df_train.merge(max_cycles, on='unit_id')
df_train['RUL'] = (df_train['max_cycle'] - df_train['time_cycle']).clip(upper=RUL_CAP)

# ── Test : on utilise les RUL ground truth fournis ─────────────────────────
# Chaque valeur = RUL au dernier cycle observé de chaque moteur
last_test    = df_test.groupby('unit_id').last().reset_index()
last_test['RUL_true'] = df_rul['RUL_true'].values

# Reconstruire le RUL pour chaque cycle du test
# cycle_from_end = max_cycle_test - time_cycle
max_cycles_test = df_test.groupby('unit_id')['time_cycle'].max().rename('max_cycle_test')
df_test = df_test.merge(max_cycles_test, on='unit_id')
df_test = df_test.merge(last_test[['unit_id', 'RUL_true']], on='unit_id')
df_test['RUL'] = (df_test['RUL_true'] + df_test['max_cycle_test'] - df_test['time_cycle']).clip(upper=RUL_CAP)

pct = (df_train['RUL'] == RUL_CAP).mean() * 100
print(f'{pct:.1f}% des lignes train plafonnées à {RUL_CAP}')
print('RUL train (5 premiers moteurs) :')
print(df_train[df_train['unit_id'] <= 2][['unit_id','time_cycle','RUL']].groupby('unit_id').tail(5))

39.4% des lignes train plafonnées à 125
RUL train (5 premiers moteurs) :
     unit_id  time_cycle  RUL
187        1         188    4
188        1         189    3
189        1         190    2
190        1         191    1
191        1         192    0
474        2         283    4
475        2         284    3
476        2         285    2
477        2         286    1
478        2         287    0


## 3. Suppression des capteurs non-informatifs + settings

In [8]:
SETTING_COLS = ['op_setting_1', 'op_setting_2', 'op_setting_3']
COLS_TO_DROP = SENSORS_TO_DROP + SETTING_COLS + ['max_cycle', 'max_cycle_test', 'RUL_true']

# Drop (ignore si colonne absente)
df_train = df_train.drop(columns=[c for c in COLS_TO_DROP if c in df_train.columns])
df_test  = df_test.drop(columns=[c for c in COLS_TO_DROP if c in df_test.columns])

FEATURE_COLS = SENSORS_TO_KEEP  # 14 capteurs

print(f'Colonnes restantes (train) : {df_train.columns.tolist()}')
print(f'Shape train : {df_train.shape}  |  Shape test : {df_test.shape}')

Colonnes restantes (train) : ['unit_id', 'time_cycle', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21', 'RUL']
Shape train : (20631, 17)  |  Shape test : (13096, 17)


## 4. Normalisation MinMaxScaler (fit sur train uniquement)

In [9]:
scaler = MinMaxScaler(feature_range=(0, 1))


df_train[FEATURE_COLS] = scaler.fit_transform(df_train[FEATURE_COLS])

# Transform sur le test (même scaler, pas de re-fit)
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])

# Sauvegarder le scaler
scaler_path = f'{OUTPUT_DIR}/minmax_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler sauvegardé → {scaler_path}')

# Vérification
print(f'\nPlage après normalisation (train) :')
print(df_train[FEATURE_COLS].agg(['min','max']).T.head(5))

Scaler sauvegardé → /content/drive/MyDrive/industrial-ai-platform/phase_preprocessing/minmax_scaler.pkl

Plage après normalisation (train) :
           min  max
sensor_02  0.0  1.0
sensor_03  0.0  1.0
sensor_04  0.0  1.0
sensor_07  0.0  1.0
sensor_08  0.0  1.0


## 5. Sliding Window (W=30)

In [10]:
WINDOW_SIZE = 30

def create_sequences(df, feature_cols, window_size=30):
    """
    Pour chaque moteur, génère des fenêtres glissantes.
    X shape : (N, window_size, n_features)
    y shape : (N,)  — RUL au dernier timestep de la fenêtre
    """
    X_list, y_list = [], []

    for unit_id, group in df.groupby('unit_id'):
        data   = group[feature_cols].values   # (T, F)
        labels = group['RUL'].values          # (T,)
        T = len(data)

        if T < window_size:
            # Moteur trop court : padding à gauche avec la première ligne
            pad    = np.repeat(data[[0]], window_size - T, axis=0)
            data   = np.vstack([pad, data])
            pad_y  = np.repeat(labels[0], window_size - T)
            labels = np.concatenate([pad_y, labels])
            T = window_size

        for t in range(T - window_size + 1):
            X_list.append(data[t : t + window_size])
            y_list.append(labels[t + window_size - 1])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def create_test_sequences(df, feature_cols, window_size=30):
    """
    Pour le test : on prend uniquement la DERNIÈRE fenêtre de chaque moteur.
    X shape : (n_engines, window_size, n_features)
    y shape : (n_engines,)
    """
    X_list, y_list = [], []

    for unit_id, group in df.groupby('unit_id'):
        data   = group[feature_cols].values
        labels = group['RUL'].values
        T = len(data)

        if T < window_size:
            pad    = np.repeat(data[[0]], window_size - T, axis=0)
            data   = np.vstack([pad, data])
            labels = np.concatenate([np.repeat(labels[0], window_size - T), labels])

        X_list.append(data[-window_size:])
        y_list.append(labels[-1])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


print(f'Sliding window W={WINDOW_SIZE} défini ✅')

Sliding window W=30 défini ✅


In [11]:
X_train, y_train = create_sequences(df_train, FEATURE_COLS, WINDOW_SIZE)
X_test,  y_test  = create_test_sequences(df_test, FEATURE_COLS, WINDOW_SIZE)

print(f'X_train : {X_train.shape}  — (samples, timesteps, features)')
print(f'y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_test  : {y_test.shape}')
print(f'\ny_train — min: {y_train.min():.1f}  max: {y_train.max():.1f}  mean: {y_train.mean():.1f}')
print(f'y_test  — min: {y_test.min():.1f}  max: {y_test.max():.1f}  mean: {y_test.mean():.1f}')

X_train : (17731, 30, 14)  — (samples, timesteps, features)
y_train : (17731,)
X_test  : (100, 30, 14)
y_test  : (100,)

y_train — min: 0.0  max: 125.0  mean: 80.6
y_test  — min: 7.0  max: 125.0  mean: 74.4


## 6. Sauvegarde des arrays + mise à jour du config

In [12]:
np.save(f'{OUTPUT_DIR}/X_train.npy', X_train)
np.save(f'{OUTPUT_DIR}/y_train.npy', y_train)
np.save(f'{OUTPUT_DIR}/X_test.npy',  X_test)
np.save(f'{OUTPUT_DIR}/y_test.npy',  y_test)

print('Fichiers sauvegardés :')
for name in ['X_train','y_train','X_test','y_test']:
    path = f'{OUTPUT_DIR}/{name}.npy'
    size = os.path.getsize(path) / 1024
    print(f'  {name}.npy  →  {size:.1f} KB')

Fichiers sauvegardés :
  X_train.npy  →  29090.0 KB
  y_train.npy  →  69.4 KB
  X_test.npy  →  164.2 KB
  y_test.npy  →  0.5 KB


In [13]:
# Mettre à jour feature_config.json avec les infos preprocessing
config.update({
    'window_size':   WINDOW_SIZE,
    'n_features':    len(FEATURE_COLS),
    'feature_cols':  FEATURE_COLS,
    'scaler':        'MinMaxScaler',
    'X_train_shape': list(X_train.shape),
    'X_test_shape':  list(X_test.shape)
})

with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print('feature_config.json mis à jour :')
print(json.dumps(config, indent=2))

feature_config.json mis à jour :
{
  "sensors_to_keep": [
    "sensor_02",
    "sensor_03",
    "sensor_04",
    "sensor_07",
    "sensor_08",
    "sensor_09",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
  ],
  "sensors_to_drop": [
    "sensor_01",
    "sensor_05",
    "sensor_06",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
  ],
  "rul_cap": 125,
  "std_threshold": 0.01,
  "corr_threshold": 0.15,
  "window_size": 30,
  "n_features": 14,
  "feature_cols": [
    "sensor_02",
    "sensor_03",
    "sensor_04",
    "sensor_07",
    "sensor_08",
    "sensor_09",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
  ],
  "scaler": "MinMaxScaler",
  "X_train_shape": [
    17731,
    30,
    14
  ],
  "X_test_shape": [
    100,
    30,
    14
  ]
}


## 📝 Résumé Preprocessing

| Étape | Décision |
|-------|----------|
| RUL label | `max_cycle - time_cycle` |
| RUL cap | 125 cycles (piecewise linear) |
| Capteurs supprimés | 7 (variance nulle ou corr < 0.15) |
| Capteurs retenus | 14 |
| Settings supprimés | 3 (variance nulle dans FD001) |
| Normalisation | MinMaxScaler [0,1] — fit train only |
| Fenêtre glissante | W = 30 cycles |
| X_train shape | (N, 30, 14) |
| X_test shape | (100, 30, 14) — 1 fenêtre par moteur |

---
*Preprocessing finalisé — prêt pour le notebook 03_LSTM.*

In [19]:
%cd '/content/drive/MyDrive/industrial-ai-platform'

/content/drive/MyDrive/industrial-ai-platform


In [20]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/chiraz-Ag/industrial-ai-platform.git

fatal: destination path 'industrial-ai-platform' already exists and is not an empty directory.


In [21]:
%cd /content
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/chiraz-Ag/industrial-ai-platform.git
!ls /content

/content
fatal: destination path 'industrial-ai-platform' already exists and is not an empty directory.
drive  industrial-ai-platform  sample_data


In [22]:
%cd /content/industrial-ai-platform
!git checkout -b dev

/content/industrial-ai-platform
Switched to a new branch 'dev'


In [25]:
import shutil, os

# Créer le dossier
os.makedirs('Data_Preprocessing_Pipeline', exist_ok=True)

# Copier le notebook
shutil.copy(
    '/content/drive/MyDrive/Colab Notebooks/Preprocessing.ipynb',
    'Data_Preprocessing_Pipeline/Preprocessing.ipynb'
)

# Copier le feature_config.json
shutil.copy(
    '/content/drive/MyDrive/industrial-ai-platform/feature_config.json',
    'Data_Preprocessing_Pipeline/feature_config.json'
)

print('Fichiers copiés ✅')
!ls Data_Preprocessing_Pipeline/

Fichiers copiés ✅
feature_config.json  Preprocessing.ipynb


In [26]:
!git config --global user.email "amirahemmar11@gmail.com"
!git config --global user.name "amirra-hmr"

!git add .
!git commit -m "feat: Phase 1 - Data Preprocessing Pipeline"
!git push origin dev

[dev a347fb3] feat: Phase 1 - Data Preprocessing Pipeline
 2 files changed, 60 insertions(+)
 create mode 100644 Data_Preprocessing_Pipeline/Preprocessing.ipynb
 create mode 100644 Data_Preprocessing_Pipeline/feature_config.json
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 6.83 KiB | 6.83 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0
remote: 
remote: Create a pull request for 'dev' on GitHub by visiting:
remote:      https://github.com/chiraz-Ag/industrial-ai-platform/pull/new/dev
remote: 
To https://github.com/chiraz-Ag/industrial-ai-platform.git
 * [new branch]      dev -> dev


In [28]:
with open('/content/industrial-ai-platform/.gitignore', 'a') as f:
    f.write('\n*.npy\n')

print('Mis à jour ✅')
!cat /content/industrial-ai-platform/.gitignore

Mis à jour ✅
data/
*.txt
*.csv
*.zip
*.h5
*.pkl
*.pt
__pycache__/
*.pyc
.ipynb_checkpoints/

*.npy


In [29]:
!git add .gitignore
!git commit -m "chore: add *.npy to gitignore"
!git push origin dev

[dev 245c808] chore: add *.npy to gitignore
 1 file changed, 2 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 396 bytes | 396.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/chiraz-Ag/industrial-ai-platform.git
   a347fb3..245c808  dev -> dev
